A. Studi Kasus: Simulasi Paralel Perambatan Partikel Polutan Sungai Lokal untuk Sistem Peringatan Dini (Early Warning System)
Latar Belakang & Masalah: Suatu kawasan industri bantaran sungai lokal mengalami kebocoran zat kimia berbahaya. Badan penanggulangan bencana daerah membutuhkan kalkulasi perkiraan konsentrasi dan waktu sebar kontaminan ke hilir pemukiman warga dalam hitungan detik. Simulasi Monte Carlo numerik pada jutaan partikel air memerlukan waktu berjam-jam jika dijalankan pada komputer tunggal (sekuensial).
Solusi: Mengadopsi prinsip HPC berupa domain/data decomposition. Beban komputasi partikel disebar secara paralel ke beberapa compute worker / core memanfaatkan pustaka paralel (concurrent.futures.ProcessPoolExecutor), kemudian dihitung akselerasinya menggunakan metrik Speedup (
𝑆
=
𝑇
seq
/
𝑇
par
S=T
seq
	​

/T
par
	​

) dan Parallel Efficiency (
𝐸
=
𝑆
𝑁
cores
×
100
%
E=
N
cores
	​

S
	​

×100%).

In [1]:
import numpy as np
import time
from concurrent.futures import ProcessPoolExecutor

# Fungsi simulasi hidrologi lokal: dispersi partikel polutan terbawa arus turbulen
def run_particle_dispersion(worker_seed, n_particles):
    np.random.seed(worker_seed)
    # Kecepatan aliran u (searah sungai) dan v (arus lateral menyamping)
    u = np.random.normal(loc=1.8, scale=0.35, size=n_particles)
    v = np.random.normal(loc=0.5, scale=0.15, size=n_particles)
    dispersion_radius = np.sqrt(u**2 + v**2)
    return np.mean(dispersion_radius), np.percentile(dispersion_radius, 95)

def studi_kasus_2_hpc():
    total_particles = 32_000_000
    n_workers = 4 # Representasi 4 Node/Core HPC
    particles_per_worker = total_particles // n_workers

    print("=== BENCHMARK SIMULASI HPC UNTUK TANTANGAN LOKAL ===")
    print(f"Total Partikel Air/Polutan yang Disimulasikan : {total_particles:,}")
    print(f"Jumlah Node/Core Paralel                      : {n_workers}")

    # 1. Eksekusi Sekuensial (Single-Core Tradisional)
    t0 = time.time()
    seq_results = [
        run_particle_dispersion(seed, particles_per_worker)
        for seed in range(n_workers)
    ]
    t_seq = time.time() - t0
    print(f"\n[Sekuensial / Single-Core] Waktu Eksekusi: {t_seq:.4f} detik")

    # 2. Eksekusi Paralel (HPC Domain Decomposition)
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        par_results = list(
            executor.map(run_particle_dispersion, range(n_workers), [particles_per_worker]*n_workers)
        )
    t_par = time.time() - t0
    print(f"[HPC Paralel / {n_workers}-Core]    Waktu Eksekusi: {t_par:.4f} detik")

    # 3. Metrik Efisiensi HPC
    speedup = t_seq / t_par if t_par > 0 else 1.0
    efficiency = (speedup / n_workers) * 100
    print(f"\nAnalisis Kinerja HPC:")
    print(f"- Speedup Factor (S)      : {speedup:.2f}x lebih cepat")
    print(f"- Parallel Efficiency (E) : {efficiency:.1f}%")

if __name__ == '__main__':
    studi_kasus_2_hpc()

=== BENCHMARK SIMULASI HPC UNTUK TANTANGAN LOKAL ===
Total Partikel Air/Polutan yang Disimulasikan : 32,000,000
Jumlah Node/Core Paralel                      : 4

[Sekuensial / Single-Core] Waktu Eksekusi: 4.5175 detik
[HPC Paralel / 4-Core]    Waktu Eksekusi: 3.4828 detik

Analisis Kinerja HPC:
- Speedup Factor (S)      : 1.30x lebih cepat
- Parallel Efficiency (E) : 32.4%
